In [ ]:
import sys
from pathlib import Path
# Add parent directory to path to import synctools modules
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import matplotlib.pyplot as plt
from speckit import noise

from synctools.frequency import FrequencyData
from synctools.clock import Clock
from synctools.synchronization import Synchronization
from synctools.signals import ThreeSignals

from speckit import compute_spectrum as lpsd
from multiprocessing import Pool
p_lpsd = {"olap":"default",
          "bmin":1,
          "Lmin":0,
          "Jdes":500,
          "Kdes":100,
          "order":2,
          "win":np.kaiser,
          "psll":250,
          "pool":Pool()}

In [ ]:

N = int(1e4) # Number of samples
noise_level = 1000 # Laser frequency noise level
nu_laser = 299792458/1064.5e-9 # Nominal laser frequency
fs = 3.3 # Sampling frequency of the phasemeters

# Generate random walk noise (red noise) for laser frequency noise
rw1 = noise.red_noise(fs, f_min=1e-3, seed=41)
rw2 = noise.red_noise(fs, f_min=1e-3, seed=42)
rw3 = noise.red_noise(fs, f_min=1e-3, seed=43)

laser1 = nu_laser - 5e6 + noise_level * rw1.get_series(N) # Laser frequency (Hz)
laser2 = nu_laser - 15e6 + noise_level * rw2.get_series(N) # Laser frequency (Hz)
laser3 = nu_laser - 10e6 + noise_level * rw3.get_series(N) # Laser frequency (Hz)

y1 = laser1 - laser2 # Phasemeter 1, PIR value (Hz)
y2 = laser3 - laser2 # Phasemeter 2, PIR value (Hz)
y3 = laser1 - laser3 # Phasemeter 3, PIR value (Hz)

np.mean(y1)/1e6, np.mean(y2)/1e6, np.mean(y3)/1e6

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3,1, figsize=(7,6), dpi=150)

ax1.plot(laser1, color='gray', linewidth=1, label='laser 1')
ax2.plot(laser2, color='gray', linewidth=1, label='laser 2')
ax3.plot(laser3, color='gray', linewidth=1, label='laser 3')
ax3.set_xlabel('Sample')
ax1.set_ylabel('Frequency (Hz)')
ax2.set_ylabel('Frequency (Hz)')
ax3.set_ylabel('Frequency (Hz)')
ax1.legend(loc='upper right', edgecolor='black', fancybox=True, shadow=True, framealpha=1)
ax2.legend(loc='upper right', edgecolor='black', fancybox=True, shadow=True, framealpha=1)
ax3.legend(loc='upper right', edgecolor='black', fancybox=True, shadow=True, framealpha=1)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7,2), dpi=150)

ax.plot(y1-y2-y3, color='black', linewidth=2)
ax.set_xlabel('Sample')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('3-signal combination for ideally synchronized phasemeters')
fig.tight_layout()

In [ ]:
shift = int(5*fs)

y1 = y1[:-2*shift]
y2 = y2[shift:-shift]
y3 = y3[shift:-shift]

shift, shift/fs

assert len(y1) == len(y2) == len(y3)

fig, ax = plt.subplots(figsize=(7,3), dpi=150)

ax.plot(y1-y2-y3, color='gray', linewidth=0.5)
ax.set_xlabel('Sample')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('3-signal combination, unsynchronized phasemeters')
fig.tight_layout()

In [ ]:
fd1 = FrequencyData(y1, fs)
fd2 = FrequencyData(y2, fs)
fd3 = FrequencyData(y3, fs)

data = [fd1, fd2, fd3]

# Create zero clock references (no clock jitter)
freqzero = FrequencyData(np.zeros(len(fd1.total)), fs)
clock12 = Clock(freqzero)
clock13 = Clock(freqzero)

data[1].register_differential_clock(clock12)
data[2].register_differential_clock(clock13)

In [ ]:
desync = ThreeSignals([fd1, fd2, fd3], p_lpsd)

In [ ]:
def combination_3sig(freqs, weights, signs=desync.signs):
	return signs[0]*weights[:,0]*freqs[:,0] + signs[1]*weights[:,1]*freqs[:,1] + signs[2]*weights[:,2]*freqs[:,2]

data_sync = Synchronization(combination_3sig, desync, fs, p_lpsd,
    model="fluc", domain="time", method="Nelder-Mead",
    interp_order=121, n_trunc=150,
    myfolder='/result_3sigs_sync/', name="3PM3S"
    )

In [ ]:
data_sync.processing(data, init_offsets=[5.,5.])

In [ ]:
fig, ax = plt.subplots(figsize=(8,3), dpi=150)

ax.plot(data_sync.freq['time'], color='black', linewidth=1)
ax.set_xlabel('Sample')
ax.set_ylabel('Frequency (Hz)')
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(5,3), dpi=150)

ax.loglog(desync.main.fourier_freq, desync.main.freq_asd, color='red', linewidth=1, label='raw')
ax.loglog(data_sync.fourier_freq, data_sync.freq['asd'], color='black', linewidth=1, label='synchronized')
ax.set_xlabel('Sample')
ax.set_ylabel('Frequency (Hz)')
fig.tight_layout()
ax.legend(loc='best', edgecolor='black', fancybox=True, shadow=True, framealpha=1)